In [4]:
from pathlib import Path


BASE_DIR = Path(".")

RAW_DIR = Path("./data/raw")
PROCESSED_DIR =  Path("./data/processed")
METADATA_DIR = Path("./metadata")

pdf_files = list(RAW_DIR.glob("*.pdf"))

print("PDF files found:")
for file in pdf_files:
    print("-", file.name)

print("\nTotal PDFs:", len(pdf_files))

PDF files found:
- aws-general.pdf
- aws-overview.pdf

Total PDFs: 2


In [3]:
!pip install pypdf


[notice] A new release of pip available: 22.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
"""
Step 2:

The idea is to read the pdfs which are stored in the data folder to read page by page
"""

In [5]:
from pypdf import PdfReader
import json



pdf_files = list(RAW_DIR.glob("*.pdf"))

all_pages = []

for pdf_path in pdf_files:
    reader = PdfReader(str(pdf_path))
    
    print(f"Reading: {pdf_path.name}")
    print(f"Total pages: {len(reader.pages)}")
    
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()
        
        if text:
            text = text.strip()
        else:
            text = ""
        
        page_data = {
            "source_file": pdf_path.name,
            "page_number": page_number,
            "text": text,
            "char_count": len(text)
        }
        
        all_pages.append(page_data)

print("\nTotal extracted pages:", len(all_pages))

Reading: aws-general.pdf
Total pages: 3579
Reading: aws-overview.pdf
Total pages: 161

Total extracted pages: 3740


In [6]:
"""
Save the extracted texts and metadata in the json file
"""

processed_file = PROCESSED_DIR / "extracted_pages.json"

with open(processed_file, "w", encoding="utf-8") as f:
    json.dump(all_pages, f, indent=4, ensure_ascii=False)

print(f"Saved extracted page data to: {processed_file}")

Saved extracted page data to: data\processed\extracted_pages.json


In [7]:
"""
Let's see the data which we have created so far
"""
for page in all_pages[:3]:
    print("=" * 80)
    print("Source:", page["source_file"])
    print("Page:", page["page_number"])
    print("Characters:", page["char_count"])
    print(page["text"][:500])

Source: aws-general.pdf
Page: 1
Characters: 134
Reference guide
AWS General Reference
Version 1.0
Copyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.
Source: aws-general.pdf
Page: 2
Characters: 562
AWS General Reference Reference guide
AWS General Reference: Reference guide
Copyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.
Amazon's trademarks and trade dress may not be used in connection with any product or service 
that is not Amazon's, in any manner that is likely to cause confusion among customers, or in any 
manner that disparages or discredits Amazon. All other trademarks not owned by Amazon are 
the property of their respective owners, who may or ma
Source: aws-general.pdf
Page: 3
Characters: 5192
AWS General Reference Reference guide
Table of Contents
AWS General Reference ................................................................................................................... 1
AWS security credential

In [ ]:
"""
Step 3: Cleaning the extracted text

The pdf text contains: extra spaces, broken lines, page headers/footers, strange newline patters, empty pages

and if we embed messy text, retrieval quality becomes poor. So this step makes the text more consistent.
"""

In [10]:
#Load extracted pages
import re

extracted_file = PROCESSED_DIR / "extracted_pages.json"

with open(extracted_file, "r", encoding="utf-8") as f:
    all_pages = json.load(f)

print("Loaded pages:", len(all_pages))

Loaded pages: 3740


In [11]:
# Create a cleaning function
def clean_text(text: str) -> str:
    if not text:
        return ""
    
    # Remove repeated whitespace
    text = re.sub(r"\s+", " ", text)
    
    # Remove spaces before punctuation
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)
    
    # Strip leading/trailing spaces
    text = text.strip()
    
    return text

In [12]:
# Apply cleaning
cleaned_pages = []

for page in all_pages:
    cleaned = clean_text(page["text"])
    
    if cleaned:
        cleaned_pages.append({
            "source_file": page["source_file"],
            "page_number": page["page_number"],
            "text": cleaned,
            "char_count": len(cleaned)
        })

print("Original pages:", len(all_pages))
print("Non-empty cleaned pages:", len(cleaned_pages))

Original pages: 3740
Non-empty cleaned pages: 3740


In [13]:
#Save cleaned pages
cleaned_file = PROCESSED_DIR / "cleaned_pages.json"

with open(cleaned_file, "w", encoding="utf-8") as f:
    json.dump(cleaned_pages, f, indent=4, ensure_ascii=False)

print("Saved cleaned pages to:", cleaned_file)

Saved cleaned pages to: data\processed\cleaned_pages.json


In [14]:
for page in cleaned_pages[:3]:
    print("=" * 80)
    print("Source:", page["source_file"])
    print("Page:", page["page_number"])
    print("Characters:", page["char_count"])
    print(page["text"][:700])

Source: aws-general.pdf
Page: 1
Characters: 134
Reference guide AWS General Reference Version 1.0 Copyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved.
Source: aws-general.pdf
Page: 2
Characters: 558
AWS General Reference Reference guide AWS General Reference: Reference guide Copyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved. Amazon's trademarks and trade dress may not be used in connection with any product or service that is not Amazon's, in any manner that is likely to cause confusion among customers, or in any manner that disparages or discredits Amazon. All other trademarks not owned by Amazon are the property of their respective owners, who may or may not be aﬃliated with, connected to, or sponsored by Amazon.
Source: aws-general.pdf
Page: 3
Characters: 5158
AWS General Reference Reference guide Table of Contents AWS General Reference....................................................................................

In [15]:
"""
Step 4: Chunk the cleaned text with metadata
A full PDF page can be too large or noisy. So we split text into smaller chunks.
each chunk should have: chunk_id, source_file, page_number, chunk_index, text, char_count
This helps the RAG system retrieve small relevant passages instead of sending entire PDFs to the LLM.
chunk_size = 500 tokens
chunk_overlap = 90 tokens
"""

'\nStep 4: Chunk the cleaned text with metadata\nA full PDF page can be too large or noisy. So we split text into smaller chunks.\neach chunk should have: chunk_id, source_file, page_number, chunk_index, text, char_count\nThis helps the RAG system retrieve small relevant passages instead of sending entire PDFs to the LLM.\n'

In [23]:
!pip install langchain-text-splitters tiktoken


[notice] A new release of pip available: 22.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
#Load cleaned pages
import hashlib
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken


with open(cleaned_file, "r", encoding="utf-8") as f:
    cleaned_pages = json.load(f)

print("Loaded cleaned pages:", len(cleaned_pages))

Loaded cleaned pages: 3740


In [26]:
#Token counter
encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(encoding.encode(text))

In [27]:
#Detect title, heading, and subheading -- Simple rule based but we can improve letter
def detect_heading(line: str):
    line = line.strip()

    if not line:
        return None

    # Remove extra numbering spaces
    normalized = re.sub(r"\s+", " ", line)

    # Example: 1. Introduction / 2.3 AWS IAM
    numbered_heading = re.match(r"^(\d+(\.\d+)*\.?)\s+(.+)", normalized)

    if numbered_heading and len(normalized) < 120:
        level = normalized.split(" ")[0].count(".") + 1
        return {
            "level": level,
            "text": normalized
        }

    # ALL CAPS headings
    if normalized.isupper() and 5 <= len(normalized) <= 100:
        return {
            "level": 1,
            "text": normalized
        }

    # Short title-like line
    if len(normalized.split()) <= 10 and len(normalized) <= 80:
        if normalized[0].isupper():
            return {
                "level": 2,
                "text": normalized
            }

    return None

In [28]:
#Convert page text into heading-aware blocks

def create_heading_blocks(cleaned_pages):
    blocks = []

    for page in cleaned_pages:
        source_file = page["source_file"]
        page_number = page["page_number"]
        text = page["text"]

        # Approx title from file name
        document_title = source_file.replace(".pdf", "").replace("-", " ").title()

        current_heading = document_title
        current_subheading = None

        # We split on sentence boundaries because earlier cleaning removed most newlines
        pseudo_lines = re.split(r"(?<=[.!?])\s+", text)

        buffer = []

        for line in pseudo_lines:
            heading_info = detect_heading(line)

            if heading_info:
                if buffer:
                    blocks.append({
                        "source_file": source_file,
                        "page_number": page_number,
                        "title": document_title,
                        "section_heading": current_heading,
                        "subheading": current_subheading,
                        "text": " ".join(buffer).strip()
                    })
                    buffer = []

                if heading_info["level"] == 1:
                    current_heading = heading_info["text"]
                    current_subheading = None
                else:
                    current_subheading = heading_info["text"]

            else:
                buffer.append(line)

        if buffer:
            blocks.append({
                "source_file": source_file,
                "page_number": page_number,
                "title": document_title,
                "section_heading": current_heading,
                "subheading": current_subheading,
                "text": " ".join(buffer).strip()
            })

    return blocks


blocks = create_heading_blocks(cleaned_pages)

print("Heading-aware blocks:", len(blocks))

Heading-aware blocks: 4813


In [33]:
TOKEN_CHUNK_SIZE = 800
TOKEN_OVERLAP = 120

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=TOKEN_CHUNK_SIZE,
    chunk_overlap=TOKEN_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [34]:
# Create chunk ID function
def create_chunk_id(source_file, page_number, chunk_index, text):
    raw = f"{source_file}-{page_number}-{chunk_index}-{text}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16]


chunks = []

for block in blocks:
    block_chunks = splitter.split_text(block["text"])

    for i, chunk_text in enumerate(block_chunks):
        chunk_text = chunk_text.strip()

        if not chunk_text:
            continue

        heading_path = block["title"]

        if block["section_heading"]:
            heading_path += " > " + block["section_heading"]

        if block["subheading"]:
            heading_path += " > " + block["subheading"]

        chunk = {
            "chunk_id": create_chunk_id(
                block["source_file"],
                block["page_number"],
                i,
                chunk_text
            ),
            "source_file": block["source_file"],
            "page_number": block["page_number"],
            "chunk_index": i,
            "title": block["title"],
            "section_heading": block["section_heading"],
            "subheading": block["subheading"],
            "heading_path": heading_path,
            "text": chunk_text,
            "token_count": count_tokens(chunk_text)
        }

        chunks.append(chunk)

print("Total chunks created:", len(chunks))

Total chunks created: 4813


In [35]:
#Create splitter
# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=1000,
#     chunk_overlap=150,
#     separators=["\n\n", "\n", ". ", " ", ""],
#     length_function=len
# )

In [36]:
#Save chunks
chunks_file = PROCESSED_DIR / "chunks_token_heading.json"

with open(chunks_file, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=4, ensure_ascii=False)

print("Saved chunks to:", chunks_file)

Saved chunks to: data\processed\chunks_token_heading.json


In [37]:
# Preview chunks
for chunk in chunks[:5]:
    print("=" * 100)
    print("Chunk ID:", chunk["chunk_id"])
    print("Source:", chunk["source_file"])
    print("Page:", chunk["page_number"])
    print("Title:", chunk["title"])
    print("Section:", chunk["section_heading"])
    print("Subheading:", chunk["subheading"])
    print("Heading Path:", chunk["heading_path"])
    print("Tokens:", chunk["token_count"])
    print(chunk["text"][:800])

Chunk ID: 9de8c6f9d1f20d0d
Source: aws-general.pdf
Page: 1
Title: Aws General
Section: Aws General
Subheading: None
Heading Path: Aws General > Aws General
Tokens: 31
Reference guide AWS General Reference Version 1.0 Copyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates.
Chunk ID: 196e9f7d41148f95
Source: aws-general.pdf
Page: 2
Title: Aws General
Section: Aws General
Subheading: None
Heading Path: Aws General > Aws General
Tokens: 32
AWS General Reference Reference guide AWS General Reference: Reference guide Copyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates.
Chunk ID: d4984591bc43463b
Source: aws-general.pdf
Page: 2
Title: Aws General
Section: Aws General
Subheading: All rights reserved.
Heading Path: Aws General > Aws General > All rights reserved.
Tokens: 84
Amazon's trademarks and trade dress may not be used in connection with any product or service that is not Amazon's, in any manner that is likely to cause confusion among customers, or in any manner that di

In [51]:
"""
This means each heading-aware block is already small enough, so the splitter did not further break most blocks.
Heading-aware blocks: 4813
Total chunks created: 4813

But before embeddings, we need to check chunk size quality.

We ideally want this:
Most chunks: 150–500 tokens
Very small chunks: not too many
Large chunks: almost zero
"""
token_counts = [chunk["token_count"] for chunk in chunks]

print("Total chunks:", len(chunks))
print("Minimum tokens:", min(token_counts))
print("Maximum tokens:", max(token_counts))
print("Average tokens:", sum(token_counts) / len(token_counts))

small_chunks = [c for c in chunks if c["token_count"] < 50]
large_chunks = [c for c in chunks if c["token_count"] > 500]

print("Small chunks < 50 tokens:", len(small_chunks))
print("Large chunks > 500 tokens:", len(large_chunks))

Total chunks: 4813
Minimum tokens: 1
Maximum tokens: 617
Average tokens: 186.7244961562435
Small chunks < 50 tokens: 832
Large chunks > 500 tokens: 29


In [39]:
"""
The only issue is too many small chunks. Very small chunks often retrieve poorly because they do not carry enough meaning alone.
"""

'\nThe only issue is too many small chunks. Very small chunks often retrieve poorly because they do not carry enough meaning alone.\n'

In [52]:
MIN_TOKENS = 80
MAX_TOKENS = 500

merged_chunks = []
buffer = None

for chunk in chunks:
    if buffer is None:
        buffer = chunk.copy()
        continue

    same_source = (
        buffer["source_file"] == chunk["source_file"]
        and buffer["page_number"] == chunk["page_number"]
        and buffer["heading_path"] == chunk["heading_path"]
    )

    combined_text = buffer["text"] + " " + chunk["text"]
    combined_tokens = count_tokens(combined_text)

    if buffer["token_count"] < MIN_TOKENS and same_source and combined_tokens <= MAX_TOKENS:
        buffer["text"] = combined_text
        buffer["token_count"] = combined_tokens
        buffer["chunk_id"] = create_chunk_id(
            buffer["source_file"],
            buffer["page_number"],
            buffer["chunk_index"],
            buffer["text"]
        )
    else:
        merged_chunks.append(buffer)
        buffer = chunk.copy()

if buffer:
    merged_chunks.append(buffer)

print("Original chunks:", len(chunks))
print("Merged chunks:", len(merged_chunks))

token_counts = [chunk["token_count"] for chunk in merged_chunks]

print("Minimum tokens:", min(token_counts))
print("Maximum tokens:", max(token_counts))
print("Average tokens:", sum(token_counts) / len(token_counts))
print("Small chunks < 50 tokens:", len([c for c in merged_chunks if c["token_count"] < 50]))
print("Large chunks > 500 tokens:", len([c for c in merged_chunks if c["token_count"] > 500]))

"""
After merge there can be some token id which can get duplicated. so we need to make it globally unique id
"""

chunks_file = PROCESSED_DIR / "chunks_token_heading_merged.json"

with open(chunks_file, "r", encoding="utf-8") as f:
    merged_chunks = json.load(f)


def create_global_chunk_id(chunk, global_index):
    raw = f"""
    {global_index}
    {chunk['source_file']}
    {chunk['page_number']}
    {chunk.get('heading_path', '')}
    {chunk['text']}
    """
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]


for global_index, chunk in enumerate(merged_chunks):
    chunk["global_chunk_index"] = global_index
    chunk["chunk_id"] = create_global_chunk_id(chunk, global_index)

# Check duplicates
ids = [chunk["chunk_id"] for chunk in merged_chunks]

print("Total IDs:", len(ids))
print("Unique IDs:", len(set(ids)))
print("Duplicate IDs:", len(ids) - len(set(ids)))

Original chunks: 4813
Merged chunks: 4754
Minimum tokens: 1
Maximum tokens: 617
Average tokens: 189.04228018510727
Small chunks < 50 tokens: 752
Large chunks > 500 tokens: 29
Total IDs: 4754
Unique IDs: 4754
Duplicate IDs: 0


In [53]:
"""
This is acceptable for now.
Merged chunks: 4754
Small chunks < 50: 752
Large chunks > 500: 29
Some small chunks are likely headings, labels, or short AWS terms. We should not over-merge blindly because it may mix unrelated topics.
"""

merged_chunks_file = PROCESSED_DIR / "chunks_token_heading_merged.json"

with open(merged_chunks_file, "w", encoding="utf-8") as f:
    json.dump(merged_chunks, f, indent=4, ensure_ascii=False)

print("Saved merged chunks to:", merged_chunks_file)

Saved merged chunks to: data\processed\chunks_token_heading_merged.json


In [42]:
"""
Step 5: Create embeddings and store in ChromaDB
Now we convert every chunk into an embedding vector.
chunk text → embedding model → numerical vector → vector database
"""

'\nStep 5: Create embeddings and store in ChromaDB\nNow we convert every chunk into an embedding vector.\nchunk text → embedding model → numerical vector → vector database\n'

In [43]:
!pip install chromadb sentence-transformers

     --------------------------------------- 23.4/23.4 MB 17.2 MB/s eta 0:00:00
     ------------------------------------- 571.3/571.3 kB 11.9 MB/s eta 0:00:00
     --------------------------------------- 12.9/12.9 MB 17.7 MB/s eta 0:00:00
     ---------------------------------------- 69.0/69.0 kB 3.7 MB/s eta 0:00:00
     -------------------------------------- 180.2/180.2 kB 3.6 MB/s eta 0:00:00
     ---------------------------------------- 60.6/60.6 kB 3.4 MB/s eta 0:00:00
     ---------------------------------------- 4.9/4.9 MB 16.4 MB/s eta 0:00:00
     -------------------------------------- 150.9/150.9 kB 9.4 MB/s eta 0:00:00
     ---------------------------------------- 2.0/2.0 MB 16.1 MB/s eta 0:00:00
     ---------------------------------------- 41.5/41.5 kB 2.1 MB/s eta 0:00:00
  Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl (8.1 MB)
     ---------------------------------------- 72.1/72.1 kB 3.9 MB/s eta 0:00:00
     ------------------------------------- 231.6/231.


[notice] A new release of pip available: 22.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [54]:
#Load merged chunks

VECTOR_DIR = "./vector_store"

chunks_file = PROCESSED_DIR / "chunks_token_heading_merged.json"

with open(chunks_file, "r", encoding="utf-8") as f:
    merged_chunks = json.load(f)

print("Chunks loaded:", len(merged_chunks))

Chunks loaded: 4754


In [ ]:
"""
Types of Embedding Model:
1. API-based (Best quality): OpenAI: text-embedding-3-small, text-embedding-3-large, Cohere, Azure OpenAI

| Model | Dim  | Cost | Quality |
| ----- | ---- | ---- | ------- |
| small | 1536 | low  | good    |
| large | 3072 | high | best    |

2. Open-source (local)
all-MiniLM-L6-v2 → fast, 384 dim
all-mpnet-base-v2 → better quality
multi-qa-* → retrieval optimized

No single embedding model is best for all tasks

"""

In [55]:
#Load local embedding model
from sentence_transformers import SentenceTransformer

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(embedding_model_name)

print("Embedding model loaded:", embedding_model_name)

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2


In [56]:
#Create ChromaDB collection
import chromadb

chroma_client = chromadb.PersistentClient(path=str("./vector_store/chroma_db"))

collection = chroma_client.get_or_create_collection(
    name="aws_docs_rag"
)

print("Collection ready:", collection.name)

Collection ready: aws_docs_rag


In [57]:
#Insert chunks in batches
BATCH_SIZE = 64

for i in range(0, len(merged_chunks), BATCH_SIZE):
    batch = merged_chunks[i:i+BATCH_SIZE]

    ids = [chunk["chunk_id"] for chunk in batch]
    texts = [chunk["text"] for chunk in batch]

    metadatas = [
        {
            "source_file": chunk["source_file"],
            "page_number": chunk["page_number"],
            "chunk_index": chunk["chunk_index"],
            "title": chunk["title"],
            "section_heading": chunk["section_heading"] or "",
            "subheading": chunk["subheading"] or "",
            "heading_path": chunk["heading_path"],
            "token_count": chunk["token_count"]
        }
        for chunk in batch
    ]

    embeddings = embedding_model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        normalize_embeddings=True
    ).tolist()

    collection.upsert(
        ids=ids,
        documents=texts,
        metadatas=metadatas,
        embeddings=embeddings
    )

    print(f"Inserted {min(i+BATCH_SIZE, len(merged_chunks))}/{len(merged_chunks)} chunks")

print("Embedding and storage completed.")

Inserted 64/4754 chunks
Inserted 128/4754 chunks
Inserted 192/4754 chunks
Inserted 256/4754 chunks
Inserted 320/4754 chunks
Inserted 384/4754 chunks
Inserted 448/4754 chunks
Inserted 512/4754 chunks
Inserted 576/4754 chunks
Inserted 640/4754 chunks
Inserted 704/4754 chunks
Inserted 768/4754 chunks
Inserted 832/4754 chunks
Inserted 896/4754 chunks
Inserted 960/4754 chunks
Inserted 1024/4754 chunks
Inserted 1088/4754 chunks
Inserted 1152/4754 chunks
Inserted 1216/4754 chunks
Inserted 1280/4754 chunks
Inserted 1344/4754 chunks
Inserted 1408/4754 chunks
Inserted 1472/4754 chunks
Inserted 1536/4754 chunks
Inserted 1600/4754 chunks
Inserted 1664/4754 chunks
Inserted 1728/4754 chunks
Inserted 1792/4754 chunks
Inserted 1856/4754 chunks
Inserted 1920/4754 chunks
Inserted 1984/4754 chunks
Inserted 2048/4754 chunks
Inserted 2112/4754 chunks
Inserted 2176/4754 chunks
Inserted 2240/4754 chunks
Inserted 2304/4754 chunks
Inserted 2368/4754 chunks
Inserted 2432/4754 chunks
Inserted 2496/4754 chunks
In

In [58]:
#Check collection count
print("Total vectors stored:", collection.count())

Total vectors stored: 6034


In [ ]:
"""
Step 6: Query → Retrieve → Answer
Now we do the reverse: User Question → embedding → vector search → top chunks → LLM answer
From your RAG doc: Query pipeline: retrieve → rerank → prompt → generate

"""

In [59]:
#Step 6.1: Query → Retrieve chunks
#Create retrieval function
def retrieve_chunks(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    retrieved = []

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    ids = results["ids"][0]
    distances = results["distances"][0]

    for doc, meta, _id, dist in zip(docs, metas, ids, distances):
        retrieved.append({
            "id": _id,
            "text": doc,
            "metadata": meta,
            "distance": dist
        })

    return retrieved

In [60]:
query = "What is AWS EC2?"

results = retrieve_chunks(query, top_k=5)

for r in results:
    print("=" * 100)
    print("Distance:", r["distance"])
    print("Source:", r["metadata"]["source_file"])
    print("Heading:", r["metadata"]["heading_path"])
    print(r["text"][:500])

Distance: 0.640029788017273
Source: aws-overview.pdf
Heading: Aws Overview > Aws Overview
Overview of Amazon Web Services AWS Whitepaper Category AWS service Cost and capacity management • AWS Savings Plan — Flexible pricing model that provides savings of up to 72% on AWS compute usage • AWS Compute Optimizer — Recommend s optimal AWS compute resources for your workloads to reduce costs and improve performance • AWS Elastic Beanstalk — Easy-to-use service for deploying and scaling web applications and services • EC2 Image Builder — Build and maintain secure Linux or Windows Server im
Distance: 0.7939727306365967
Source: aws-overview.pdf
Heading: Aws Overview > Aws Overview
Overview of Amazon Web Services AWS Whitepaper Conclusion AWS provides building blocks that you can assemble quickly to support virtually any workload. With AWS, you’ll ﬁnd a complete set of highly available services that are designed to work together to build sophisticated scalable applications. You have access to h

In [61]:
def build_context(retrieved_chunks):
    context = ""

    for i, chunk in enumerate(retrieved_chunks, 1):
        meta = chunk["metadata"]

        context += f"""
[chunk_{i}]
Source: {meta['source_file']}
Heading: {meta['heading_path']}
Text:
{chunk['text']}

"""
    return context

In [62]:
!pip install openai


[notice] A new release of pip available: 22.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [63]:
from openai import OpenAI

client = OpenAI(api_key="sk-...")

In [64]:
def ask_rag(query):
    retrieved = retrieve_chunks(query, top_k=5)
    context = build_context(retrieved)

    prompt = f"""
You are an AWS expert assistant.

Answer ONLY using the provided context.
If the answer is not present, say "Not found in provided documents."

Question:
{query}

Context:
{context}
"""

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt,
        temperature=0.1,
        max_output_tokens=500
    )

    return response.output_text, retrieved

In [65]:
query = "Explain AWS EC2 in simple terms"

answer, retrieved = ask_rag(query)

print("\nANSWER:\n")
print(answer)


ANSWER:

Amazon EC2 (Amazon Elastic Compute Cloud) is a web service that provides secure, resizable compute capacity in the cloud. It allows you to rent virtual servers to run your applications, paying only for the compute capacity you actually use. This makes it easy to scale your computing resources up or down as needed.


In [ ]:
"""
Step 7: Add Reranking
Right now Chroma gives us chunks based on vector similarity.
But vector retrieval is only the first filter.
Query → retrieve top 20 chunks → rerank them → keep best 5 chunks
Embedding retrieval is fast but sometimes approximate. A reranker reads the query and each chunk together, then gives a relevance score.
So instead of directly using top 5 from Chroma, we retrieve more chunks first: top_k = 20
Then rerank and keep: top_n = 5

"""

In [66]:
!pip install sentence-transformers


[notice] A new release of pip available: 22.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [67]:
from sentence_transformers import CrossEncoder

reranker_model_name = "cross-encoder/ms-marco-MiniLM-L6-v2"

reranker = CrossEncoder(reranker_model_name)

print("Reranker loaded:", reranker_model_name)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Reranker loaded: cross-encoder/ms-marco-MiniLM-L6-v2


In [68]:
def rerank_chunks(query, retrieved_chunks, top_n=5):
    pairs = []

    for chunk in retrieved_chunks:
        pairs.append((query, chunk["text"]))

    scores = reranker.predict(pairs)

    reranked = []

    for chunk, score in zip(retrieved_chunks, scores):
        new_chunk = chunk.copy()
        new_chunk["rerank_score"] = float(score)
        reranked.append(new_chunk)

    reranked = sorted(
        reranked,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked[:top_n]

In [69]:
#Test retrieval + reranking
query = "What is AWS EC2?"

retrieved = retrieve_chunks(query, top_k=20)

reranked = rerank_chunks(query, retrieved, top_n=5)

for r in reranked:
    print("=" * 100)
    print("Rerank Score:", r["rerank_score"])
    print("Vector Distance:", r["distance"])
    print("Source:", r["metadata"]["source_file"])
    print("Page:", r["metadata"]["page_number"])
    print("Heading:", r["metadata"]["heading_path"])
    print(r["text"][:700])

Rerank Score: 6.545708179473877
Vector Distance: 0.640029788017273
Source: aws-overview.pdf
Page: 44
Heading: Aws Overview > Aws Overview
Overview of Amazon Web Services AWS Whitepaper Category AWS service Cost and capacity management • AWS Savings Plan — Flexible pricing model that provides savings of up to 72% on AWS compute usage • AWS Compute Optimizer — Recommend s optimal AWS compute resources for your workloads to reduce costs and improve performance • AWS Elastic Beanstalk — Easy-to-use service for deploying and scaling web applications and services • EC2 Image Builder — Build and maintain secure Linux or Windows Server images • Elastic Load Balancing (ELB) — Automatic ally distribute incoming application traﬃc across multiple targets Amazon EC2 Amazon Elastic Compute Cloud (Amazon EC2) is a web service that provides s
Rerank Score: 4.652578353881836
Vector Distance: 0.8197640776634216
Source: aws-overview.pdf
Page: 42
Heading: Aws Overview > Aws Overview
Overview of Amazon Web

In [70]:
#Update RAG function to use reranking
def ask_rag_with_reranking(query):
    retrieved = retrieve_chunks(query, top_k=20)
    reranked = rerank_chunks(query, retrieved, top_n=5)

    context = build_context(reranked)

    prompt = f"""
You are an AWS expert assistant.

Answer ONLY using the provided context.
If the answer is not present, say "Not found in provided documents."

Cite the chunk numbers like [chunk_1], [chunk_2].

Question:
{query}

Context:
{context}
"""

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt,
        temperature=0.1,
        max_output_tokens=600
    )

    return response.output_text, reranked

In [71]:
#Test final reranked RAG
query = "Explain AWS EC2 in simple terms"

answer, reranked_chunks = ask_rag_with_reranking(query)

print("ANSWER:")
print(answer)

print("\nUSED CHUNKS:")
for i, chunk in enumerate(reranked_chunks, 1):
    print("=" * 100)
    print(f"chunk_{i}")
    print("Score:", chunk["rerank_score"])
    print("Source:", chunk["metadata"]["source_file"])
    print("Page:", chunk["metadata"]["page_number"])
    print("Heading:", chunk["metadata"]["heading_path"])

ANSWER:
Amazon Elastic Compute Cloud (Amazon EC2) is a web service that provides secure, resizable compute capacity in the cloud. It allows you to run virtual servers on demand, giving you the flexibility to scale your computing resources up or down as needed [chunk_1].

USED CHUNKS:
chunk_1
Score: 2.282684326171875
Source: aws-overview.pdf
Page: 44
Heading: Aws Overview > Aws Overview
chunk_2
Score: 2.161458969116211
Source: aws-overview.pdf
Page: 49
Heading: Aws Overview > Aws Overview
chunk_3
Score: 1.9916131496429443
Source: aws-overview.pdf
Page: 54
Heading: Aws Overview > Aws Overview > With Amazon ECR, there are no upfront fees or commitments.
chunk_4
Score: 1.865794062614441
Source: aws-overview.pdf
Page: 109
Heading: Aws Overview > Aws Overview
chunk_5
Score: 1.767534852027893
Source: aws-overview.pdf
Page: 147
Heading: Aws Overview > Aws Overview


In [ ]:
#Prev: Chroma top 5 → LLM
#Now: Chroma top 20 → CrossEncoder reranker → best 5 → LLM